# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer Exploration with `mlcroissant`

This notebook demonstrates how to load and explore the FAIR² dataset (Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors) using the [`mlcroissant`](https://mlcommons.github.io/croissant/) library.

### Dataset Source
The dataset is described via a [Croissant schema](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json), providing machine-actionable metadata for tabular clinical-patient data.

In [ ]:
# Install the mlcroissant library if needed
!pip install mlcroissant

## 1. Data Loading

Use `mlcroissant` to load dataset metadata and records.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Instantiate the Dataset object
dataset = mlc.Dataset(croissant_url)

# Display dataset name and description from metadata
print(f"{dataset.metadata.name}: {dataset.metadata.description}")


## 2. Data Overview

List all available record set `@id`s and, for each, the field `@id`s and their labels. This will help us reference entities by their Croissant `@id` in later steps.

In [ ]:
# Show all record set @id's with associated field @id's and labels

metadata = dataset.metadata  # Convenient alias

# Extract record sets - these usually reside in metadata.record_sets
record_sets = getattr(metadata, 'record_sets', [])
if not record_sets:
    print("No record sets found! Please check the schema.")
else:
    for recset in record_sets:
        print(f"RecordSet @id: {recset['@id']}")
        fields = recset.get('fields', [])
        if not fields:
            print('  No fields for this record set.')
        else:
            for field in fields:
                label = field.get('name', '<no label>')
                print(f"  Field @id: {field['@id']}, Label: {label}")
        print('---')


## 3. Data Extraction

Load data from each record set using the record set and field `@id` values investigated above. Data are loaded into pandas DataFrames, stored in a dictionary indexed by the record set `@id`.

In [ ]:
# Collect all record set @id values
record_set_ids = []
if hasattr(metadata, 'record_sets'):
    record_set_ids = [rs['@id'] for rs in metadata.record_sets]
else:
    print('No record sets found!')

# Attempt to load each record set as a DataFrame
dataframes = {}
for recset_id in record_set_ids:
    print(f"Loading records for RecordSet: {recset_id}")
    try:
        records = list(dataset.records(record_set=recset_id))
        dataframes[recset_id] = pd.DataFrame(records)
        print(f"Loaded {len(records)} records.")
    except Exception as e:
        print(f"Could not load record set {recset_id}: {e}")

# Show columns for the first available record set DataFrame
if dataframes:
    first_rs_id = list(dataframes.keys())[0]
    print(f"\nColumns for record set {first_rs_id}:")
    print(dataframes[first_rs_id].columns.tolist())
    dataframes[first_rs_id].head()
else:
    print("No dataframes loaded!")



## 4. Exploratory Data Analysis (EDA)

We now filter records by a numeric field (e.g. age, or interval-between-cancers if available), normalize it, and optionally group by a categorical field (such as sex or anatomical location). 
- All references to specific fields use their `@id`.


In [ ]:
# Choose the first available record set for EDA
if not dataframes:
    print("No DataFrames to analyze.")
else:
    rs_id = list(dataframes.keys())[0]
    df = dataframes[rs_id]

    # Substitute with actual Croissant @id for age field (or similar numeric field)
    # Candidates (examples): '@id': 'age', '@id': 'interval_between_cancers'
    # We'll attempt to select a numeric (int/float) column present
    numeric_field_id = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break
    if numeric_field_id is None:
        print("No numeric field found for EDA.")
    else:
        print(f"Using numeric field: {numeric_field_id} (@id reference)")

        threshold = df[numeric_field_id].quantile(0.75)  # Use 75th percentile as illustrative threshold
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())

        # Normalize the selected numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        )
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try to group by a suitable categorical field (e.g. sex, location, or similar)
        group_field_id = None
        for col in df.columns:
            # Use a non-numeric, non-index field as possible group (string categories)
            if col != numeric_field_id and pd.api.types.is_object_dtype(df[col]):
                group_field_id = col
                break
        if group_field_id is not None:
            grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
            print(f"\nMean {numeric_field_id} grouped by {group_field_id} (@id):")
            print(grouped)
        else:
            print("No suitable categorical field for grouping found.")


## 5. Visualization

Plot the distribution of a selected numeric field, and grouped means if available. Visualization leverages matplotlib and seaborn (install if missing).

In [ ]:
# Visualization Section
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes:
    rs_id = list(dataframes.keys())[0]
    df = dataframes[rs_id]

    # Use the same numeric field as in EDA, if found
    numeric_field_id = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break
    if numeric_field_id:
        plt.figure(figsize=(6, 4))
        sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=15)
        plt.title(f"Distribution of {numeric_field_id} (@id)")
        plt.xlabel(numeric_field_id)
        plt.ylabel("Count")
        plt.show()

        # If a categorical field is suitable, show boxplot
        group_field_id = None
        for col in df.columns:
            if col != numeric_field_id and pd.api.types.is_object_dtype(df[col]):
                group_field_id = col
                break
        if group_field_id:
            plt.figure(figsize=(8, 4))
            sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
            plt.title(f"{numeric_field_id} by {group_field_id} (@id)")
            plt.xlabel(group_field_id)
            plt.ylabel(numeric_field_id)
            plt.xticks(rotation=45)
            plt.tight_layout()
            plt.show()
    else:
        print("No suitable numeric field found for visualization.")
else:
    print("No dataframes to visualize.")


## 6. Conclusion

- This notebook demonstrates loading, inspection, and basic EDA for a biomedical Croissant dataset using `mlcroissant`.
- We accessed and filtered fields exclusively by their `@id` values, enabling precise programmatic referencing.
- For custom analysis or further processing, always refer to dataset elements by their `@id` as mandated in the Croissant specification.

**Thank you for using FAIR² and mlcroissant!**